# 菲米尼 LLM 伺服器

在 server 電腦用 VS Code 打開這個檔案，由上而下按 ▶ 執行。

- **第一次**：1 → 2 → 3 → 4 → 5
- **之後每次**：1 → 5

要先裝好 Python，VS Code 裝好 Python 和 Jupyter 擴充功能。

## 1. 設定（每次都要先執行）

In [ ]:
import ipaddress
import json
import os
import socket
import subprocess
import tempfile
import time
import urllib.request
from pathlib import Path

MODEL = "qwen3:14b"  # 想換模型改這裏
OLLAMA = Path(os.environ["LOCALAPPDATA"]) / "Programs" / "Ollama" / "ollama.exe"
URL = "http://127.0.0.1:11434"
RULE = "Ollama LLM Server"
LOG = Path(tempfile.gettempdir()) / "ollama_server.log"


def ollama_up():
    try:
        urllib.request.urlopen(URL, timeout=2)
        return True
    except OSError:
        return False


def wait_up(seconds=60):
    for _ in range(seconds):
        if ollama_up():
            return True
        time.sleep(1)
    return False


def post(path, body, timeout=600):
    req = urllib.request.Request(URL + path, json.dumps(body).encode("utf-8"),
                                 {"Content-Type": "application/json"})
    return urllib.request.urlopen(req, timeout=timeout)


print("設定完成，模型：", MODEL)

## 2. 安裝 Ollama（只做一次）

In [ ]:
if OLLAMA.exists():
    print("Ollama 已經安裝")
else:
    setup = Path(tempfile.gettempdir()) / "OllamaSetup.exe"
    shown = set()

    def progress(blocks, size, total):
        pct = min(100, blocks * size * 100 // total) // 10 * 10 if total > 0 else 0
        if pct not in shown:
            shown.add(pct)
            print(f"下載 Ollama… {pct}%")

    urllib.request.urlretrieve("https://ollama.com/download/OllamaSetup.exe", setup, progress)
    print("安裝中，請等一下…")
    subprocess.run([str(setup), "/SILENT"], check=True)
    if not OLLAMA.exists():
        raise RuntimeError(f"安裝後找不到 {OLLAMA}")
    print("安裝完成")

## 3. 下載模型（只做一次，`qwen3:14b` 約 9GB）

In [ ]:
if not ollama_up():
    subprocess.Popen([str(OLLAMA), "serve"], creationflags=subprocess.CREATE_NO_WINDOW,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not wait_up():
        raise RuntimeError("Ollama 60 秒內沒有啟動")

shown = set()
with post("/api/pull", {"model": MODEL}, timeout=None) as r:
    for line in r:
        d = json.loads(line)
        if "error" in d:
            raise RuntimeError(d["error"])
        if d.get("total"):
            # 只在每 10% 印一次，不然會印幾千行
            pct = d.get("completed", 0) * 100 // d["total"] // 10 * 10
            key = (d.get("digest"), pct)
            if d["total"] > 100_000_000 and key not in shown:
                shown.add(key)
                print(f"下載中… {pct}%")
        elif d.get("status") == "success":
            print(f"{MODEL} 下載完成")

## 4. 開防火牆（只做一次）

會彈出「是否允許變更」的視窗，按 **是**。

只讓同一個網絡和 Tailscale（`100.64.0.0/10`）的電腦連入。不分私人／公用網絡，因為 Windows 新連的 Wi-Fi 預設是公用。指定了 Ollama 程式，Windows 就不會彈出自己的防火牆視窗（那個視窗按錯會封鎖 Ollama）。

In [ ]:
# 防火牆要管理員權限：把指令寫進暫存檔，再用管理員身份執行它
script = Path(tempfile.gettempdir()) / "ollama_firewall.cmd"
script.write_text(
    f'netsh advfirewall firewall delete rule name="{RULE}"\r\n'
    f'netsh advfirewall firewall add rule name="{RULE}" dir=in action=allow program="{OLLAMA}" '
    f'protocol=TCP localport=11434 remoteip=localsubnet,100.64.0.0/10 profile=any\r\n',
    encoding="ascii")
subprocess.run(["powershell", "-NoProfile", "-Command",
                "Start-Process -FilePath $env:FW_SCRIPT -Verb RunAs -Wait -WindowStyle Hidden"],
               env={**os.environ, "FW_SCRIPT": str(script)}, check=True)

check = subprocess.run(["netsh", "advfirewall", "firewall", "show", "rule", f"name={RULE}"],
                       capture_output=True)
print("防火牆已開放" if check.returncode == 0 else "防火牆規則沒有加到，是不是按了「否」？再執行一次")

## 5. 開伺服器（每次用都要開）

執行後會顯示 IP，然後一直運行。

- **要停止**：按這個 cell 左邊的停止鍵 ■
- 直接關掉 VS Code 的話伺服器**不會停**，要再開這個檔案執行 1 和 5，然後按 ■

In [ ]:
# 開機自動開的 Ollama 只聽本機，先關掉，再用下面的設定重開
for name in ("ollama app.exe", "ollama.exe"):
    subprocess.run(["taskkill", "/f", "/im", name], capture_output=True)
time.sleep(2)

env = {**os.environ,
       "OLLAMA_HOST": "0.0.0.0:11434",   # 讓其他電腦連得到
       "OLLAMA_KEEP_ALIVE": "-1",        # 模型一直留在顯存
       "OLLAMA_CONTEXT_LENGTH": "8192"}
log = open(LOG, "w", encoding="utf-8")
server = subprocess.Popen([str(OLLAMA), "serve"], env=env, stdout=log, stderr=subprocess.STDOUT,
                          creationflags=subprocess.CREATE_NO_WINDOW)
if not wait_up():
    server.kill()
    raise RuntimeError(f"Ollama 60 秒內沒有啟動，記錄在 {LOG}")

tailscale = ipaddress.ip_network("100.64.0.0/10")
for ip in sorted({a[4][0] for a in socket.getaddrinfo(socket.gethostname(), None, socket.AF_INET)}):
    if ipaddress.ip_address(ip) in tailscale:
        print(f"Tailscale IP：{ip}   ← 填到聊天電腦 config/llm.json 的 server.base_url")
    elif not ip.startswith("169.254."):
        print(f"區域網絡 IP：{ip}")

print("\n載入模型並測試…")
t = time.time()
with post("/v1/chat/completions", {
        "model": MODEL, "max_tokens": 100, "reasoning_effort": "none",
        "messages": [{"role": "user", "content": "用一句話打招呼"}]}) as r:
    reply = json.loads(r.read())["choices"][0]["message"]["content"]
print(f"測試回覆（{time.time() - t:.0f} 秒）：{reply.strip()}")

print("\n伺服器運行中。要停止：按這個 cell 左邊的 ■")
try:
    server.wait()
    print(f"伺服器自己停了，記錄在 {LOG}")
except KeyboardInterrupt:
    server.terminate()
    server.wait()
    print("伺服器已停止")
finally:
    log.close()